# Acoustic ID — Module 3: Signal Decoder
### Acoustic ID-Based Vehicle Horn Detection for Urban Noise Pollution Control

---

**Scope of this module:**
- Reads noisy `.wav` files from `noisy_signals/` (Module 2B output)
- Runs the full signal processing decode pipeline on each file
- Returns one of four detection modes per file:
  - `ACOUSTIC_ID` — full decode success, vehicle positively identified
  - `ACOUSTIC_ID_CORRUPTED` — preamble found but CRC failed
  - `ANPR_FALLBACK` — no preamble found (tampered/unregistered module)
  - `UNREGISTERED_MODULE` — CRC passed but ID not in database
- Updates `acoustic_id.db` with decode status, timestamp, and context flag per vehicle
- Runs the 7×4 reliability test and prints the accuracy table for the paper

**Horn detection note:** In real hardware, the audible microphone channel (separate from
the ultrasonic channel) runs a CNN horn classifier that triggers the decode pipeline.
In simulation, every file in `noisy_signals/` represents one already-triggered horn event.
Horn detection is a hardware concern and is not simulated here.

**Libraries:** `numpy`, `scipy`, `soundfile`, `sqlite3` — no new installs required.

## Step 1: Import Libraries

In [ ]:
import numpy as np
import soundfile as sf
import sqlite3
import os
import glob
import hashlib
from datetime import datetime
from scipy import signal as sp_signal

## Step 2: Configuration

**Only the first two lines need to change between runs.**

All signal constants must match Module 2 and Module 2B exactly.

**Context flag threshold:** If ambient noise in a decoded file exceeds
`AMBIENT_REVIEW_THRESHOLD_DB`, the event is flagged `CONTEXTUAL_REVIEW`
rather than `NORMAL`. Module 4 routes `CONTEXTUAL_REVIEW` events to human
review instead of auto-issuing a challan — this is Solution 2 (ambient noise
protection for innocent drivers).

**NCC threshold:** The normalised cross-correlation score required to confirm
preamble detection. Values below this threshold produce `ANPR_FALLBACK`.
0.3 is robust across all tested SNR conditions.

In [ ]:
# ── USER CONFIGURATION — change only these two lines ─────────────────────────
DB_FILE     = "acoustic_id.db"    # Module 1 database
NOISY_DIR   = "noisy_signals"     # Module 2B output folder
# ─────────────────────────────────────────────────────────────────────────────

# Signal constants — must match Module 2 and Module 2B
SAMPLE_RATE     = 96000
CARRIER_FREQ    = 25000
BIT_DURATION    = 0.005
PREAMBLE        = "1010"
GUARD_SILENCE   = 0.010
SAMPLES_PER_BIT = int(SAMPLE_RATE * BIT_DURATION)

# Decoder parameters
NCC_THRESHOLD               = 0.3   # minimum NCC score to confirm preamble
AMBIENT_REVIEW_THRESHOLD_DB = 85    # dB above which event routes to human review

# Reliability test parameters
PROFILES  = ["traffic", "rain", "dj", "procession", "hawker", "crowd", "tamper"]
DB_LEVELS = [60, 70, 80, 90]

notebook_dir = os.getcwd()
db_path      = os.path.join(notebook_dir, DB_FILE)
noisy_dir    = os.path.join(notebook_dir, NOISY_DIR)

print(f"Database     : '{db_path}'")
print(f"Noisy signals: '{noisy_dir}'")

## Step 3: CRC-8

Copied from Module 1. Used in the decode pipeline to validate the 8-bit checksum
appended to every 32-bit Acoustic ID before transmission.

In [ ]:
def compute_crc8(data_bits: str) -> str:
    """
    Compute CRC-8 checksum for a binary string.
    Polynomial: 0x07 (standard CRC-8).

    Args:
        data_bits (str): Binary string of any length

    Returns:
        str: 8-bit CRC as binary string
    """
    padded     = data_bits.zfill(((len(data_bits) + 7) // 8) * 8)
    byte_array = int(padded, 2).to_bytes(len(padded) // 8, byteorder="big")
    crc = 0
    for byte in byte_array:
        crc ^= byte
        for _ in range(8):
            crc = ((crc << 1) ^ 0x07) if (crc & 0x80) else (crc << 1)
            crc &= 0xFF
    return format(crc, "08b")

## Step 4: Database Connection and Schema Extension

Opens the existing `acoustic_id.db` from Module 1 and adds three new columns
to the `vehicles` table. Safe to run multiple times — columns already present
are silently skipped.

| Column | Type | Purpose |
|---|---|---|
| `last_decode_status` | TEXT | Most recent decode outcome for this vehicle |
| `last_decode_time` | TEXT | Timestamp of most recent decode attempt |
| `context_flag` | TEXT | `NORMAL` or `CONTEXTUAL_REVIEW` — read by Module 4 |

In [ ]:
def setup_module3(db_path: str) -> sqlite3.Connection:
    """
    Open Module 1 database and extend vehicles table with decode result columns.

    Args:
        db_path (str): Path to acoustic_id.db

    Returns:
        sqlite3.Connection: Open connection with row_factory set to sqlite3.Row

    Raises:
        FileNotFoundError: If database does not exist
    """
    if not os.path.exists(db_path):
        raise FileNotFoundError(
            f"Database not found: '{db_path}'\n"
            "Run Module 1 first to generate acoustic_id.db"
        )

    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row

    for col, definition in [
        ("last_decode_status", "TEXT    DEFAULT NULL"),
        ("last_decode_time",   "TEXT    DEFAULT NULL"),
        ("context_flag",       "TEXT    DEFAULT NULL"),
    ]:
        try:
            conn.execute(f"ALTER TABLE vehicles ADD COLUMN {col} {definition}")
            conn.commit()
        except sqlite3.OperationalError as e:
            if "duplicate column" not in str(e).lower():
                raise

    return conn


conn = setup_module3(db_path)
print(f"Database ready: '{db_path}'")

## Step 5: Signal Processing Utilities

**`bandpass_filter`** — 4th-order Butterworth bandpass filter in SOS form,
20–45 kHz. Isolates the ultrasonic band and discards all audible content.
After this step, any noise below 20 kHz (traffic, music, crowd, rain) is gone.

**`detect_envelope`** — rectifies the filtered signal and applies a low-pass
filter at twice the bit rate (400 Hz). Converts the 25 kHz carrier into a smooth
energy curve where bit `1` appears as a raised plateau and bit `0` as near-zero.

**`measure_ambient_db`** — RMS dB of the full input signal. Used to set
the context flag after decoding.

In [ ]:
def bandpass_filter(
    signal_in : np.ndarray,
    sr        : int = SAMPLE_RATE
) -> np.ndarray:
    """
    4th-order Butterworth bandpass filter, 20–45 kHz, SOS form.
    Isolates ultrasonic OOK signal and discards all audible noise.

    Args:
        signal_in (np.ndarray): Raw audio from .wav file
        sr        (int)       : Sample rate in Hz

    Returns:
        np.ndarray: float32 filtered signal
    """
    nyq = sr / 2.0
    sos = sp_signal.butter(
        4,
        [np.clip(20000 / nyq, 1e-4, 0.9999),
         np.clip(45000 / nyq, 1e-4, 0.9999)],
        btype="band",
        output="sos"
    )
    return sp_signal.sosfilt(sos, signal_in.astype(np.float64)).astype(np.float32)


def detect_envelope(
    filtered : np.ndarray,
    sr       : int = SAMPLE_RATE
) -> np.ndarray:
    """
    Rectify and low-pass filter to extract OOK amplitude envelope.

    Low-pass cutoff = 2 × bit rate = 400 Hz — smooths within each
    5ms bit window while preserving the on/off switching pattern.

    Args:
        filtered (np.ndarray): Bandpass-filtered ultrasonic signal
        sr       (int)       : Sample rate in Hz

    Returns:
        np.ndarray: float32 envelope, peak-normalised to [0, 1]
    """
    rectified = np.abs(filtered).astype(np.float64)
    cutoff    = np.clip((2.0 / BIT_DURATION) / (sr / 2.0), 1e-4, 0.9999)
    sos_lp    = sp_signal.butter(4, cutoff, btype="low", output="sos")
    envelope  = sp_signal.sosfilt(sos_lp, rectified)
    peak      = np.max(np.abs(envelope))
    return (envelope / peak if peak > 0 else envelope).astype(np.float32)


def measure_ambient_db(signal: np.ndarray) -> float:
    """
    Compute RMS dB of a signal.

    Args:
        signal (np.ndarray): Audio signal

    Returns:
        float: RMS level in dB, or -inf if silent
    """
    rms = np.sqrt(np.mean(signal.astype(np.float64) ** 2))
    return float(20 * np.log10(rms)) if rms > 1e-12 else float("-inf")

## Step 6: Preamble Search

Locates the `1010` synchronisation sequence in the envelope using
normalised cross-correlation (NCC).

**Why NCC instead of raw correlation:**
Raw correlation output scales with signal amplitude — the threshold would need
to change with noise level. NCC normalises output to `[-1, 1]` regardless of
signal energy, so a single threshold of `0.3` works reliably across all SNR conditions.

If the best NCC score across the entire signal does not exceed `NCC_THRESHOLD`,
the function returns `-1` and the decoder classifies the event as `ANPR_FALLBACK`.
This is the tamper detection path — a horn signal with no OOK above 20 kHz
will never produce an NCC score above threshold.

In [ ]:
def find_preamble(
    envelope : np.ndarray,
    sr       : int = SAMPLE_RATE
) -> int:
    """
    Search for the OOK preamble (1010) in the amplitude envelope
    using normalised cross-correlation.

    Returns the sample index where the transmission ID begins
    (immediately after the preamble), or -1 if not found.

    Args:
        envelope (np.ndarray): Peak-normalised amplitude envelope
        sr       (int)       : Sample rate in Hz

    Returns:
        int: Transmission start index, or -1 if preamble not found
    """
    spb      = SAMPLES_PER_BIT
    n_bits   = len(PREAMBLE)

    # Build ideal preamble template: 1=high, 0=low
    template = np.zeros(n_bits * spb, dtype=np.float64)
    for i, bit in enumerate(PREAMBLE):
        if bit == "1":
            template[i * spb : (i + 1) * spb] = 1.0

    n_t       = len(template)
    n_s       = len(envelope)
    if n_s < n_t:
        return -1

    tmpl_mean = np.mean(template)
    tmpl_std  = np.std(template)
    if tmpl_std < 1e-12:
        return -1

    best_ncc = -2.0
    best_idx = -1
    stride   = max(1, spb // 4)  # check every quarter-bit for efficiency

    for i in range(0, n_s - n_t + 1, stride):
        window = envelope[i : i + n_t].astype(np.float64)
        w_mean = np.mean(window)
        w_std  = np.std(window)
        if w_std < 1e-12:
            continue
        ncc = np.mean((window - w_mean) * (template - tmpl_mean)) / (w_std * tmpl_std)
        if ncc > best_ncc:
            best_ncc = ncc
            best_idx = i

    if best_ncc < NCC_THRESHOLD:
        return -1

    tx_start = best_idx + n_t
    if tx_start + 40 * spb > n_s:
        return -1

    return tx_start

## Step 7: OOK Demodulator

Once the preamble position is known, the decoder measures the mean envelope
energy in each consecutive 5ms window. Above the threshold = bit `1`,
below = bit `0`. Threshold is set adaptively at 50% of the global envelope mean.

Produces a 40-bit binary string (32-bit acoustic ID + 8-bit CRC).

In [ ]:
def demodulate_ook(
    envelope  : np.ndarray,
    start_idx : int,
    sr        : int = SAMPLE_RATE
) -> str | None:
    """
    Decode 40 OOK bits starting at start_idx in the envelope.

    Threshold = 50% of global envelope mean (adaptive — adjusts
    automatically to signal level after bandpass + envelope stages).

    Args:
        envelope  (np.ndarray): Peak-normalised amplitude envelope
        start_idx (int)       : Sample index where transmission ID begins
        sr        (int)       : Sample rate in Hz

    Returns:
        str : 40-bit binary string, or None if signal ends before 40 bits
    """
    spb       = SAMPLES_PER_BIT
    threshold = np.mean(envelope) * 0.5
    bits      = []

    for i in range(40):  # 32-bit acoustic ID + 8-bit CRC
        ws = start_idx + i * spb
        we = ws + spb
        if we > len(envelope):
            return None
        energy = np.mean(envelope[ws:we])
        bits.append("1" if energy > threshold else "0")

    return "".join(bits)

## Step 8: Full Decode Pipeline

`decode_wav()` runs the complete pipeline on a single `.wav` file and
returns a structured result dict.

**Pipeline stages:**
1. Load WAV — verify sample rate
2. Measure ambient dB → set `context_flag`
3. Bandpass filter 20–45 kHz
4. Detect amplitude envelope
5. NCC preamble search
6. OOK demodulate 40 bits
7. CRC-8 validate
8. Database lookup

**`plate_hint`** — simulates the ANPR camera. In the `ANPR_FALLBACK` path
(no preamble found), the plate is extracted from the filename and used to
look up the vehicle. In real hardware this would come from the camera's
OCR output at the moment of horn detection.

In [ ]:
def decode_wav(
    wav_path   : str,
    conn       : sqlite3.Connection,
    plate_hint : str = None
) -> dict:
    """
    Run the full decode pipeline on a .wav file.

    Args:
        wav_path   (str)                : Path to noisy .wav file
        conn       (sqlite3.Connection) : Open DB connection
        plate_hint (str)                : Plate number from filename
                                         (simulates ANPR camera output)

    Returns:
        dict: {
            "wav_path"       : str,
            "plate_hint"     : str or None,
            "detection_mode" : "ACOUSTIC_ID" | "ACOUSTIC_ID_CORRUPTED" |
                               "ANPR_FALLBACK" | "UNREGISTERED_MODULE" | "ERROR",
            "decoded_id"     : str or None  -- 40-bit binary string if found,
            "crc_passed"     : bool or None,
            "db_match"       : bool,
            "vehicle"        : dict or None -- full vehicle record if matched,
            "context_flag"   : "NORMAL" | "CONTEXTUAL_REVIEW",
            "ambient_db"     : float,
            "error"          : str or None
        }
    """
    result = {
        "wav_path"       : wav_path,
        "plate_hint"     : plate_hint,
        "detection_mode" : None,
        "decoded_id"     : None,
        "crc_passed"     : None,
        "db_match"       : False,
        "vehicle"        : None,
        "context_flag"   : None,
        "ambient_db"     : None,
        "error"          : None
    }

    # ── Stage 1: Load ────────────────────────────────────────────────────────
    if not os.path.exists(wav_path):
        result["detection_mode"] = "ERROR"
        result["error"]          = f"File not found: '{wav_path}'"
        return result

    try:
        signal_in, sr = sf.read(wav_path, dtype="float32")
    except Exception as e:
        result["detection_mode"] = "ERROR"
        result["error"]          = f"Failed to read WAV: {e}"
        return result

    if sr != SAMPLE_RATE:
        result["detection_mode"] = "ERROR"
        result["error"]          = f"Sample rate {sr} Hz — expected {SAMPLE_RATE} Hz"
        return result

    # ── Stage 2: Ambient noise measurement ───────────────────────────────────
    ambient_db             = measure_ambient_db(signal_in)
    result["ambient_db"]   = round(ambient_db, 2)
    result["context_flag"] = (
        "CONTEXTUAL_REVIEW"
        if ambient_db >= AMBIENT_REVIEW_THRESHOLD_DB
        else "NORMAL"
    )

    # ── Stage 3: Bandpass filter ──────────────────────────────────────────────
    filtered = bandpass_filter(signal_in, sr)

    # ── Stage 4: Envelope detection ───────────────────────────────────────────
    envelope = detect_envelope(filtered, sr)

    # ── Stage 5: Preamble search ──────────────────────────────────────────────
    tx_start = find_preamble(envelope, sr)

    if tx_start == -1:
        # No preamble found — tampered module or total noise corruption
        result["detection_mode"] = "ANPR_FALLBACK"
        if plate_hint:
            row = conn.execute(
                "SELECT * FROM vehicles WHERE plate_number = ?", (plate_hint,)
            ).fetchone()
            if row:
                result["vehicle"]  = dict(row)
                result["db_match"] = True
        return result

    # ── Stage 6: OOK demodulation ─────────────────────────────────────────────
    tx_bits = demodulate_ook(envelope, tx_start, sr)

    if tx_bits is None:
        result["detection_mode"] = "ANPR_FALLBACK"
        return result

    result["decoded_id"] = tx_bits

    # ── Stage 7: CRC validation ───────────────────────────────────────────────
    acoustic_id  = tx_bits[:32]
    received_crc = tx_bits[32:]

    if received_crc != compute_crc8(acoustic_id):
        result["detection_mode"] = "ACOUSTIC_ID_CORRUPTED"
        result["crc_passed"]     = False
        return result

    result["crc_passed"] = True

    # ── Stage 8: Database lookup ──────────────────────────────────────────────
    row = conn.execute(
        "SELECT * FROM vehicles WHERE acoustic_id = ?", (acoustic_id,)
    ).fetchone()

    if not row:
        result["detection_mode"] = "UNREGISTERED_MODULE"
        return result

    result["detection_mode"] = "ACOUSTIC_ID"
    result["vehicle"]        = dict(row)
    result["db_match"]       = True
    return result

## Step 9: Database Update

Updates the vehicle's record in `acoustic_id.db` after decoding.
These three columns are read by Module 4 to decide the challan path.

Called automatically by `run_reliability_test()` for every decoded file.

In [ ]:
def update_decode_result(
    plate        : str,
    status       : str,
    context_flag : str,
    conn         : sqlite3.Connection
) -> None:
    """
    Write decode outcome back to the vehicles table.

    Args:
        plate        (str): Vehicle plate number
        status       (str): Detection mode string
        context_flag (str): "NORMAL" or "CONTEXTUAL_REVIEW"
        conn (sqlite3.Connection): Active DB connection
    """
    conn.execute(
        """
        UPDATE vehicles
        SET last_decode_status = ?,
            last_decode_time   = ?,
            context_flag       = ?
        WHERE plate_number = ?
        """,
        (status, datetime.now().isoformat(timespec="seconds"), context_flag, plate)
    )
    conn.commit()

## Step 10: Single File Decode

Decode one file manually to verify the pipeline is working before running
the full reliability test. Change the filename as needed.

In [ ]:
# ── Decode one file — change filename to inspect any specific case ───────────
import glob as _glob

_files = sorted(_glob.glob(os.path.join(noisy_dir, "*_acoustic_id.wav"))
                + _glob.glob(os.path.join(noisy_dir, "*.wav")))

if not _files:
    print(f"No .wav files found in '{noisy_dir}'")
    print("Run Module 2B first to generate noisy signal files.")
else:
    # Default: decode the first file found
    SAMPLE_FILE = os.path.basename(_files[0])

    wav_path = os.path.join(noisy_dir, SAMPLE_FILE)
    # Extract plate hint from filename: PLATE_PROFILE_NNdb.wav
    parts       = SAMPLE_FILE.replace(".wav", "").split("_")
    plate_hint  = "_".join(parts[:-2]) if len(parts) >= 3 else None

    result = decode_wav(wav_path, conn, plate_hint=plate_hint)

    print(f"File            : {SAMPLE_FILE}")
    print(f"Detection mode  : {result['detection_mode']}")
    print(f"Context flag    : {result['context_flag']}")
    print(f"Ambient dB      : {result['ambient_db']}")
    print(f"CRC passed      : {result['crc_passed']}")
    print(f"DB match        : {result['db_match']}")
    if result["vehicle"]:
        print(f"Plate           : {result['vehicle']['plate_number']}")
        print(f"Owner           : {result['vehicle']['owner_name']}")
    if result["error"]:
        print(f"Error           : {result['error']}")

## Step 11: Reliability Test

`run_reliability_test()` decodes all `.wav` files in `noisy_signals/`,
aggregates results into a 7×4 matrix, and updates the database after
each successful decode.

**For additive profiles** (traffic, rain, dj, procession, hawker, crowd):
counts `ACOUSTIC_ID` decodes as success.

**For the tamper profile:**
counts `ANPR_FALLBACK` as success — this is the correct outcome for a
tampered module. 100% ANPR_FALLBACK on tamper files means the system
correctly detects every tampered vehicle at every noise level.

In [ ]:
def run_reliability_test(
    noisy_dir : str,
    conn      : sqlite3.Connection
) -> dict:
    """
    Decode all noisy WAV files and build the 7x4 accuracy matrix.

    Args:
        noisy_dir (str)                : Path to noisy_signals/ folder
        conn      (sqlite3.Connection) : Active DB connection

    Returns:
        dict: Nested {profile: {db_level: {metric: count}}}
    """
    if not os.path.exists(noisy_dir):
        print(f"Folder not found: '{noisy_dir}'")
        print("Run Module 2B first.")
        return {}

    results = {
        profile: {
            db: {"attempts": 0, "acoustic_id": 0, "anpr_fallback": 0,
                 "corrupted": 0, "unregistered": 0, "error": 0}
            for db in DB_LEVELS
        }
        for profile in PROFILES
    }

    wav_files = sorted(glob.glob(os.path.join(noisy_dir, "*.wav")))
    if not wav_files:
        print(f"No .wav files in '{noisy_dir}'. Run Module 2B first.")
        return results

    for wav_path in wav_files:
        fname = os.path.basename(wav_path)
        parts = fname.replace(".wav", "").split("_")
        if len(parts) < 3:
            continue

        db_str  = parts[-1].replace("db", "")
        profile = parts[-2]
        plate   = "_".join(parts[:-2])

        try:
            db = int(db_str)
        except ValueError:
            continue

        if profile not in PROFILES or db not in DB_LEVELS:
            continue

        result = decode_wav(wav_path, conn, plate_hint=plate)
        results[profile][db]["attempts"] += 1
        mode = result["detection_mode"]

        if   mode == "ACOUSTIC_ID"          : results[profile][db]["acoustic_id"]   += 1
        elif mode == "ANPR_FALLBACK"         : results[profile][db]["anpr_fallback"] += 1
        elif mode == "ACOUSTIC_ID_CORRUPTED" : results[profile][db]["corrupted"]     += 1
        elif mode == "UNREGISTERED_MODULE"   : results[profile][db]["unregistered"]  += 1
        else                                 : results[profile][db]["error"]         += 1

        # Update DB for successfully identified vehicles
        if result["vehicle"] and result["context_flag"]:
            update_decode_result(
                plate        = result["vehicle"]["plate_number"],
                status       = mode,
                context_flag = result["context_flag"],
                conn         = conn
            )

    return results

## Step 12: Run Reliability Test and Print Accuracy Table

Runs the full test across all vehicles and profiles, then prints
the 7×4 table for the paper.

**SNR conditions:**
60 dB = SNR +20 dB (light noise) → 70 dB = SNR +10 dB → 80 dB = SNR 0 dB → 90 dB = SNR −20 dB (extreme)

**Tamper row** shows % ANPR_FALLBACK (correct tamper detection) not acoustic decode accuracy.

In [ ]:
results = run_reliability_test(noisy_dir, conn)

if results:
    # ── Print accuracy table ──────────────────────────────────────────────────
    col_w = 12
    header = (
        f"{'Profile':<14}"
        f"  {'60dB (+20dB SNR)':>{col_w}}"
        f"  {'70dB (+10dB SNR)':>{col_w}}"
        f"  {'80dB (0dB SNR)':>{col_w}}"
        f"  {'90dB (-20dB SNR)':>{col_w}}"
    )
    print("ACOUSTIC ID DECODE ACCURACY")
    print("=" * len(header))
    print(header)
    print("-" * len(header))

    for profile in PROFILES:
        row_vals = []
        for db in DB_LEVELS:
            r = results[profile][db]
            if r["attempts"] == 0:
                row_vals.append(f"{'N/A':>{col_w}}")
                continue
            if profile == "tamper":
                # Tamper: success = ANPR_FALLBACK
                pct = r["anpr_fallback"] / r["attempts"] * 100
                row_vals.append(f"{pct:>{col_w-1}.0f}%*")
            else:
                pct = r["acoustic_id"] / r["attempts"] * 100
                row_vals.append(f"{pct:>{col_w-1}.0f}%")
        label = profile.capitalize()
        print(f"{label:<14}  {'  '.join(row_vals)}")

    print("-" * len(header))
    print("* Tamper column = % ANPR_FALLBACK (correct tamper detection rate)")
    print()

    # ── Summary counts ────────────────────────────────────────────────────────
    total_attempts = sum(
        results[p][db]["attempts"]
        for p in PROFILES for db in DB_LEVELS
    )
    total_acoustic = sum(
        results[p][db]["acoustic_id"]
        for p in PROFILES if p != "tamper" for db in DB_LEVELS
    )
    total_tamper_correct = sum(
        results["tamper"][db]["anpr_fallback"] for db in DB_LEVELS
    )
    print(f"Total files processed  : {total_attempts}")
    print(f"Acoustic ID success    : {total_acoustic}")
    print(f"Tamper correctly flagged: {total_tamper_correct}")

## Step 13: Close Database Connection

Module 4 opens its own connection to `acoustic_id.db`.

In [ ]:
conn.close()